In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import numpy as np

# Styling for better presentation
plt.style.use('seaborn-v0_8-muted')

In [ ]:
def run_simulation(initial_fish, algae_count, light_mode, filter_power, days):
    # Initial chemical states
    o2, co2, nh3 = 100, 50, 10
    current_fish = initial_fish
    high_filter_days = 0
    data = []

    # Constants for biological processes
    # Purple light is modeled as the most efficient for O2 production
    light_eff = {'Purple (Max)': 2.5, 'White (Normal)': 1.0, 'Green (Low)': 0.1}

    # Filter settings based on power levels
    # Format: [Ammonia reduction rate, Stress level description]
    filter_settings = {
        'High (Max Clean)': [2.0, 'High'],
        'Medium (Balanced)': [0.8, 'Normal'],
        'Low (Weak)': [0.2, 'Low']
    }

    for t in range(days + 1):
        f_reduction, stress = filter_settings[filter_power]
        eff = light_eff[light_mode]

        # 1. Algae Production (Numerical step: State changes over time)
        algae_o2 = algae_count * 0.6 * eff
        algae_co2 = algae_count * 0.4 * eff
        algae_nh3 = algae_count * 0.15 * eff

        # 2. Fish Consumption and Waste
        fish_o2 = current_fish * 0.5
        fish_co2 = current_fish * 0.4
        fish_nh3 = current_fish * 0.25

        # 3. Dynamic Death Mechanisms
        death_count = 0

        # Scenario A: High Power Stress (Death every 2 days after day 5)
        if filter_power == 'High (Max Clean)':
            high_filter_days += 1
            if high_filter_days > 5 and t % 2 == 0:
                death_count += 1
        else:
            high_filter_days = 0

        # Scenario B: Low Power & Ammonia Toxicity
        if filter_power == 'Low (Weak)' and nh3 > 80:
            death_count += 1

        current_fish = max(0, current_fish - death_count)

        # 4. Updating the environment (Discrete-time update)
        o2 = max(0, min(200, o2 + algae_o2 - fish_o2))
        co2 = max(0, min(200, co2 + fish_co2 - algae_co2))
        nh3 = max(0, min(200, nh3 + fish_nh3 - algae_nh3 - f_reduction))

        data.append({
            'Day': t, 'O2': o2, 'CO2': co2, 'NH3': nh3,
            'Fish': current_fish
        })

    return pd.DataFrame(data), filter_power, light_mode

In [ ]:
def plot_results(df, filter_p, light_m):
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Plotting Chemicals
    ax1.set_xlabel('Days')
    ax1.set_ylabel('Chemical Concentration', color='black')
    ax1.plot(df['Day'], df['O2'], label='Oxygen (O2)', color='deepskyblue', lw=2)
    ax1.plot(df['Day'], df['CO2'], label='CO2', color='salmon')
    ax1.plot(df['Day'], df['NH3'], label='Ammonia (NH3)', color='limegreen', ls='--')
    ax1.legend(loc='upper left')
    ax1.grid(alpha=0.3)

    # Plotting Fish Population on secondary Y-axis
    ax2 = ax1.twinx()
    ax2.set_ylabel('Fish Count', color='blue', fontsize=12, fontweight='bold')
    ax2.step(df['Day'], df['Fish'], label='Fish Population', color='blue', where='post', lw=3)
    ax2.tick_params(axis='y', labelcolor='blue')

    plt.title(f"Ecosystem Evolution\nLight: {light_m} | Filter: {filter_p}", fontsize=14)
    plt.show()

# Link the simulation and plot
def interactive_hub(fish, algae, light, filter, days):
    results_df, f_mode, l_mode = run_simulation(fish, algae, light, filter, days)
    plot_results(results_df, f_mode, l_mode)

In [ ]:
interact(interactive_hub,
         fish=widgets.IntSlider(min=1, max=40, value=15, description='Fish:'),
         algae=widgets.IntSlider(min=0, max=40, value=10, description='Algae:'),
         light=['Purple (Max)', 'White (Normal)', 'Green (Low)'],
         filter=['High (Max Clean)', 'Medium (Balanced)', 'Low (Weak)'],
         days=widgets.IntSlider(min=1, max=30, value=20, description='Days:'))